# ⚽ AI Football T4 — Player ID & Highlights
### Kaggle T4 GPU Notebook


In [ ]:
# Setup
!git clone https://github.com/jordiagi/ai-football-t4
%cd ai-football-t4
!pip install -r requirements.txt -q

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# CONFIG
VIDEO_PATH = '/kaggle/input/your-dataset/match.mp4'
TARGET_PLAYER = 'Marc'    # Name for output files
TARGET_JERSEY = 10        # Jersey number (optional, improves ID)
TARGET_TEAM = 'home'      # 'home' or 'away' (based on shirt color)
DETECT_STRIDE = 2         # Detect every N frames (1=all, 2=every other)
CHUNK_MINUTES = 5         # Process in 5-min chunks for checkpointing

In [ ]:
# PHASE 1: Detection + Tracking (~2-3hr for 90min match)
from src.pipeline import run_detection_phase
tracks = run_detection_phase(
    video_path=VIDEO_PATH,
    stride=DETECT_STRIDE,
    chunk_minutes=CHUNK_MINUTES,
    checkpoint_dir='checkpoints/'
)
print(f'Tracked {len(tracks)} unique IDs')

In [ ]:
# PHASE 2: Player Identification (~1hr)
from src.identify import run_identification
player_map = run_identification(
    tracks=tracks,
    video_path=VIDEO_PATH,
    target_name=TARGET_PLAYER,
    target_jersey=TARGET_JERSEY,
    target_team=TARGET_TEAM
)
print('Player map:', player_map)

In [ ]:
# PHASE 3: Highlight Export
from src.export import run_export
run_export(
    video_path=VIDEO_PATH,
    player_map=player_map,
    target_name=TARGET_PLAYER,
    output_dir='output/'
)
print('Done! Check output/ folder')